# Lesson 2 : LangGraph Components

### 本节课的核心思路：用 LangGraph 重写 Lesson 1 的 ReAct Agent

Lesson 1 里我们手写了 `while` 循环 + 正则解析来实现 ReAct，这一课改用 LangGraph 的核心组件来做同样的事：

- **State（状态）**：一个 `TypedDict`，描述图在运行过程中一直传递、累积的数据（这里是消息列表 `messages`）
- **Node（节点）**：图里的一个处理步骤，本质是一个「输入 state，返回要合并进 state 的更新」的函数（比如调用 LLM 的节点、执行工具的节点）
- **Edge（边）**：节点之间的固定跳转关系
- **Conditional Edge（条件边）**：根据当前 state 的内容，动态决定下一步走向哪个节点（比如：模型要不要调用工具）
- **Tool calling（工具调用）**：不再像 Lesson 1 那样自己写正则去解析 "Action: xxx: yyy"，而是用 LLM 原生支持的 function/tool calling 能力，模型直接返回结构化的 `tool_calls`

这些组件组合起来，就是 LangGraph 版本的 ReAct 循环：LLM 节点 -> 判断是否要调用工具（条件边）-> 工具节点 -> 回到 LLM 节点，直到模型不再请求调用工具为止。

In [ ]:
from dotenv import load_dotenv
_ = load_dotenv()  # 可选：本方案改用本地 Ollama + DuckDuckGo，已不再需要 OPENAI_API_KEY / TAVILY_API_KEY

# ==== 本地无密钥模型配置（Ollama + Qwen）====
# 前提：安装 Ollama（https://ollama.com）并执行 `ollama pull qwen2.5`
OLLAMA_BASE_URL = "http://localhost:11434/v1"  # Ollama 的 OpenAI 兼容端点
OLLAMA_API_KEY = "ollama"                        # 占位符，Ollama 不校验密钥
MODEL = "qwen2.5"                                # 中文模型；多跳推理可换更大的，如 qwen2.5:14b

In [ ]:
from langgraph.graph import StateGraph,END   # StateGraph：构建状态图；END 是一个特殊节点，表示图执行结束
from typing import TypedDict,Annotated
import operator
from langchain_core.messages import AnyMessage,SystemMessage,HumanMessage,ToolMessage
from langchain_openai import ChatOpenAI
# 【改为无密钥搜索】原课程用 Tavily（需申请 TAVILY_API_KEY）。这里改用 DuckDuckGo：
# 无需任何密钥即可联网搜索。需先 `pip install duckduckgo-search`（或 ddgs）。
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

In [ ]:
# 用 DuckDuckGo 搜索替代 Tavily：max_results 限制每次最多返回 4 条结果
_search_wrapper = DuckDuckGoSearchAPIWrapper(max_results=4)
tool = DuckDuckGoSearchResults(api_wrapper=_search_wrapper)
print(type(tool))
print(tool.name)  # 工具的 name（duckduckgo_results_json）会被注册进 LLM 的 tool schema，模型靠这个名字来"点名"调用哪个工具

In [ ]:
class AgentState(TypedDict):
    # messages 是图的状态：Annotated[..., operator.add] 表示每个节点返回的 messages 列表
    # 会用 "+"（列表拼接）累加进已有状态，而不是覆盖——这样每一轮 LLM/工具的输出都会被保留下来，
    # 而不是每次都把历史消息冲掉
    messages:Annotated[list[AnyMessage],operator.add]


In [ ]:
class Agent:
    def __init__(self,model,tools,system=""):
        self.system=system
        graph=StateGraph(AgentState)                     # 用 AgentState 作为图的状态 schema 创建一个状态图
        # 【bug 修复】原来写的是 self.cal_openai（typo，少了一个 l），
        # 但下面定义的方法名是 call_openai，Python 找不到 self.cal_openai 这个属性，
        # 实例化 Agent 时会直接抛出 AttributeError: 'Agent' object has no attribute 'cal_openai'。
        # 修复：改成 self.call_openai，与实际定义的方法名保持一致。
        graph.add_node("llm",self.call_openai)            # 添加名为 "llm" 的节点，对应调用大模型这一步
        graph.add_node("action",self.take_action)         # 添加名为 "action" 的节点，对应执行工具这一步
        graph.add_conditional_edges(
            "llm",
            self.exists_action,       # 条件函数：根据 llm 节点的输出判断要不要走向 action 节点
            {True: "action", False: END}  # 条件函数返回 True 就跳到 action，返回 False 就直接结束
        )
        graph.add_edge("action", "llm")   # 工具执行完之后，固定边：回到 llm 节点，让模型看 Observation 继续推理
        graph.set_entry_point("llm")      # 图的入口节点是 llm（第一步永远先让模型思考）
        self.graph = graph.compile()      # 编译成可执行的图对象
        self.tools = {t.name: t for t in tools}      # 工具名 -> 工具对象的映射，等价于 Lesson 1 里的 know_actions
        self.model = model.bind_tools(tools)         # 把工具 schema 绑定到模型上，模型才知道有哪些工具可用、参数长什么样
    def exists_action(self, state: AgentState):
        result = state['messages'][-1]         # 取出最新一条消息（应该是模型刚返回的 AIMessage）
        return len(result.tool_calls) > 0      # 判断模型这一轮是否请求了工具调用（tool_calls 非空即代表要执行 Action）

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages   # 每次调用都把 system prompt 拼在最前面
        message = self.model.invoke(messages)
        return {'messages': [message]}   # 返回值会被 operator.add 追加进状态里的 messages 列表

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls   # 拿到模型请求的所有工具调用（可能一次请求多个工具）
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            if not t['name'] in self.tools:      # check for bad tool name from LLM
                print("\n ....bad tool name....")
                result = "bad tool name, retry"  # instruct LLM to retry if bad
            else:
                result = self.tools[t['name']].invoke(t['args'])   # 真正执行工具，t['args'] 是模型给出的结构化参数
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
            # ToolMessage 必须带上 tool_call_id，这样模型才能把这条 Observation 和它自己发起的那次调用对应起来
        print("Back to the model!")
        return {'messages': results}

In [ ]:
prompt = """你是一个聪明的研究助手。请使用搜索引擎来查找信息。\
你可以多次调用搜索（可以一次并行发起多个，也可以分几轮依次调用）。\
只在你清楚自己要查什么的时候才去搜索。\
如果在追问之前需要先查一些信息，你也可以先查再问！请用中文回答最终结果。
"""
# 创建底层聊天模型：指向本地 Ollama（无需密钥）。构造对象本身不发请求，真正调用在 graph.invoke 时才发生
model=ChatOpenAI(model=MODEL, base_url=OLLAMA_BASE_URL, api_key=OLLAMA_API_KEY)
abot=Agent(model,[tool],system=prompt)    # 组装出带工具调用能力的 Agent：模型 + 工具列表 + 系统提示词

In [ ]:
from IPython.display import Image
# 【运行环境问题修复】原代码 draw_png() 依赖本地安装 pygraphviz（还需要系统级 Graphviz），
# 当前 venv 没装这个可选依赖，直接调用会抛 ImportError: Install pygraphviz to draw graphs。
# draw_mermaid_png() 虽然不需要 pygraphviz，但需要联网请求 mermaid.ink 生成图片，同样跑不通。
# 这里改用 draw_mermaid()：纯本地生成 Mermaid 图定义文本，不依赖任何额外的包或网络，
# 把这段文本复制到支持 Mermaid 的编辑器（比如 VS Code、Mermaid Live Editor）里就能看到可视化的流程图。
# 如果你本地装好了 pygraphviz，也可以把下面这行换回 Image(abot.graph.get_graph().draw_png())。
print(abot.graph.get_graph().draw_mermaid())

In [ ]:
messages = [HumanMessage(content="旧金山现在天气怎么样？")]
result = abot.graph.invoke({"messages": messages})  # 运行整个图：llm -> (判断要不要调用工具) -> action -> llm -> ... 直到没有 tool_calls 为止

In [ ]:
result['messages'][-1].content  # 图执行完毕后，最后一条消息就是模型给出的最终回答

In [ ]:
messages = [HumanMessage(content="旧金山和洛杉矶现在的天气怎么样？")]
result = abot.graph.invoke({"messages": messages})  # 这次问题涉及两个城市，模型可能在一轮里发起多个并行的 tool_calls

In [ ]:
result['messages'][-1].content

In [ ]:
# 注意：为得到更一致的结果，这里对问题做了改写；每次运行结果可能因搜索内容与模型不同而变化。
query = "2024 年的超级碗（Super Bowl）是谁赢的？夺冠球队的总部位于哪个州？该州的 GDP 是多少？请分别回答每个问题。"
messages = [HumanMessage(content=query)]

# 多跳推理（需要连续查多次不同信息）对模型能力要求更高，建议换用更大的本地模型（如先 `ollama pull qwen2.5:14b`）
model = ChatOpenAI(model=MODEL, base_url=OLLAMA_BASE_URL, api_key=OLLAMA_API_KEY)
abot = Agent(model, [tool], system=prompt)  # 重新创建 Agent（新模型 + 空白的对话历史）
result = abot.graph.invoke({"messages": messages})

In [ ]:
print(result['messages'][-1].content)